# Forecast pipeline: preprocessing + modelling in one notebook (test shop + Sets A, B, C, D)

This notebook goes from the raw parquet files to a submission CSV in one run. It replaces the two-notebook setup ([preprocessing.ipynb](preprocessing.ipynb) → [modelling.ipynb](modelling.ipynb)), which stays as it was, and adds:

| New | Why |
|---|---|
| **Set B** is loaded, cleaned and ready for training | requested. It made the score slightly worse, so it is off for training by default: add `'B'` to `TRAIN_SOURCES` in §12 to use it (§16) |
| **Stock features** from the test shop's `on_hand` | stock on hand right after a launch shows whether an article is live |
| **Launch features** (sales since the first sale) | new articles decide the score (see below) |
| **All young-article origins in training** (`YOUNG_WEEKS`) | there are few launches, and every one of them is costly in the metric |
| **A separate model for young articles** (`young_lgb`) | the main model forecasts 0 for every brand-new article; this one learns their sales level |
| **Rounding to whole units** on by default | improved every learner on every backtest window |
| **Error by article age** (§14) | shows where points are won and lost |

**The key fact about this metric.** MASE divides each article's error by its *average week-to-week change over its whole history*. A newly launched article has ~130 weeks of zeros before its first sale, so that number is tiny (one sale in 130 weeks → 0.015), and every unit of error counts ~70×. In the backtest, the ~80 articles launched in the last 13 weeks cost **25–45 points**, while the other ~2,800 articles together score **+30**. Getting the young articles right matters more than anything else.

**How it is organised.** Every step is a small function. Pipelines are lists of steps, so to change something, **edit only the configuration cells** (§6 sources, §8 features, §12 models): reorder, comment out, or change options.

| § | What |
|---|---|
| 1 | Setup and the pipeline runner |
| 2–5 | Preprocessing functions: loaders, cleaning, daily → weekly, panel steps |
| 6 | **Configure the sources** and build the weekly panel |
| 7–8 | Feature functions, **configure the features**, build the feature frame, sanity check |
| 9 | Train / validation sets and the metric |
| 10–11 | Models and post-processing |
| 12 | **Configure the models** |
| 13–15 | Rolling backtest, error by article age, overfitting check |
| 16 | Which training sources help (Set B on/off; optional, slow) |
| 17–19 | Feature importance, final fit + submission, score chart |

**Runtime:** about 30–40 minutes on a 16-core laptop (§16 adds ~45 minutes when switched on). It needs ~8 GB of RAM.

**To run:** Run All. The submission is written to `submissions/submission_full_<model>.csv`.

In [ ]:
%pip install -q numpy pandas matplotlib pyarrow scikit-learn lightgbm

## 1. Setup and the pipeline runner

In [ ]:
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from IPython.display import display
from matplotlib.patches import Patch
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

# Works whether the kernel starts in the notebook folder or the project root
BASE = Path('.') if Path('set-a').exists() else Path('data/hackathon-retail-sets')
SUBMISSION_DIR = BASE / 'submissions'

H = 13                              # forecast horizon (w1..w13)
PAIR = ['store_id', 'article_id']
SID = 'series_id'
SEED = 0

BLUE, GREY, INK = '#2a78d6', '#b5b4ae', '#52514e'
plt.rcParams.update({
    'figure.figsize': (11, 4), 'figure.dpi': 100,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': GREY, 'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK,
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
})


def _shape(obj):
    if isinstance(obj, pd.DataFrame):
        return f'{len(obj):,} rows x {obj.shape[1]} cols'
    return type(obj).__name__


def run_pipeline(data, steps, name='', verbose=True):
    """Apply `steps` in order. A step is `f` or `(f, {options})`; `None` entries are skipped."""
    for step in steps:
        if step is None:
            continue
        func, kw = (step, {}) if callable(step) else (step[0], dict(step[1]) if len(step) > 1 else {})
        t0 = time.perf_counter()
        data = func(data, **kw)
        if verbose:
            print(f'[{name}] {func.__name__:<26s} -> {_shape(data):<26s} ({time.perf_counter() - t0:.1f}s)')
    return data


def to_week(dates):
    """Monday of the week (the test shop's week_start convention)."""
    return (dates - pd.to_timedelta(dates.dt.day_of_week, unit='D')).dt.normalize()

## 2. Loaders

One per set. They fix data types and return one table; nothing is filtered here.

- **Test shop**: article × week with `on_hand` (end-of-week stock, noisy).
- **Set A**: store × article × week with the article hierarchy.
- **Set B** (synthetic): basket lines joined to their baskets → daily store × article units. Stock and promotion tables are fixed in `load_set_b_tables` (store ids, timestamps, duplicate stock events) but not used for features, because the test shop has no promo data.
- **Set C**: `Decimal` numbers and `date` objects converted.
- **Set D**: daily sales lines.

In [ ]:
def load_test_shop(base=BASE):
    df = pd.read_parquet(base / 'test-shop' / 'history.parquet')
    df['week_start'] = pd.to_datetime(df['week_start'])
    df['on_hand'] = df['on_hand'].astype(float)
    return df.sort_values(['article_id', 'week_start'], ignore_index=True)


def load_set_a(base=BASE, with_articles=True, drop_cols=('format',)):
    df = pd.read_parquet(base / 'set-a' / 'weekly.parquet')
    df['week_start'] = pd.to_datetime(df['week_start'])
    if with_articles:
        art = pd.read_parquet(base / 'set-a' / 'articles.parquet').drop(columns=list(drop_cols))
        df = df.merge(art, on=PAIR, how='left', validate='many_to_one')
    return df.sort_values([*PAIR, 'week_start'], ignore_index=True)


def load_set_b_tables(base=BASE):
    """All Set B tables with the known issues fixed (DATASETS §4)."""
    t = {p.stem: pd.read_parquet(p) for p in sorted((base / 'set-b').glob('*.parquet'))}
    t['baskets']['timestamp'] = pd.to_datetime(t['baskets']['timestamp'])
    t['stock_events']['timestamp'] = pd.to_datetime(t['stock_events']['timestamp'])
    for c in ['start_date', 'end_date']:
        t['promotions'][c] = pd.to_datetime(t['promotions'][c])
    t['stock']['store_id'] = t['stock']['store_id'].str.lstrip('S').astype(int)   # 'S0001' -> 1, like the baskets
    t['stock_events'] = t['stock_events'].drop_duplicates(ignore_index=True)     # 181 exact duplicates
    return t


def load_set_b(base=BASE, with_articles=True):
    """Daily store x article units from the basket lines (transactions only, like Set D)."""
    t = load_set_b_tables(base)
    lines = t['sales_lines'].merge(t['baskets'][['transaction_id', 'store_id', 'timestamp']],
                                   on='transaction_id', how='left', validate='many_to_one')
    lines['date'] = lines['timestamp'].dt.normalize()
    daily = lines.groupby(['date', *PAIR], as_index=False)['units'].sum()
    if with_articles:
        daily = daily.merge(t['articles'][['article_id', 'category', 'subcategory']], on='article_id', how='left')
    return daily.sort_values([*PAIR, 'date'], ignore_index=True)


C_NUM = ['is_return', 'units', 'sales_amount', 'cogs', 'discount_amount', 'discount_pct', 'on_hand',
         'unit_cost', 'unit_price', 'latest_on_hand', 'days_since_receipt', 'days_since_sale']


def load_set_c(base=BASE):
    df = pd.read_parquet(base / 'set-c' / 'daily.parquet')
    df[C_NUM] = df[C_NUM].astype(float)
    df['is_return'] = df['is_return'].astype(int)
    for c in ['date', 'last_receipt_date', 'last_sold_date']:
        df[c] = pd.to_datetime(df[c])
    return df.sort_values([*PAIR, 'date'], ignore_index=True)


def load_set_d(base=BASE):
    df = pd.read_parquet(base / 'set-d' / 'sales.parquet')
    df['date'] = pd.to_datetime(df['date'])
    return df.sort_values([*PAIR, 'date'], ignore_index=True)

## 3. Source-specific cleaning steps

| Step | Used for | Why |
|---|---|---|
| `ts_flag_censored` | test shop | 0 units **and** `on_hand ≤ 0` the week before = possibly a stock-out |
| `a_flag_out_of_stock` | A | `in_stock = False` means sales were capped |
| `drop_pairs` | C | the PROD-0000202 burst at STR-022 (292 units in one day) |
| `drop_columns` | C, D | leaky columns in C (`is_phasing_out`, same-day `days_since_sale`, …), constant ones in D |
| `handle_returns` | C, D | net returns (default) or ignore them |
| `filter_sales_types` | C, D | optional: leave out e.g. clearance sales |

In [ ]:
def ts_flag_censored(df, max_on_hand=0):
    prev = df.groupby('article_id')['on_hand'].shift(1)
    return df.assign(censored=(df['units'] == 0) & (prev <= max_on_hand))


def a_flag_out_of_stock(df):
    return df.assign(censored=~df['in_stock'].astype(bool))


C_OUTLIER_PAIRS = [('STR-022', 'PROD-0000202')]
C_LEAKY_COLS = ['is_phasing_out', 'latest_on_hand', 'on_hand', 'days_since_sale', 'last_sold_date']
D_CONSTANT_COLS = ['sales_channel', 'store_type', 'return_reason']


def drop_pairs(df, pairs, keys=PAIR):
    bad = pd.MultiIndex.from_tuples(pairs, names=keys)
    return df[~pd.MultiIndex.from_frame(df[keys]).isin(bad)].reset_index(drop=True)


def drop_columns(df, cols):
    return df.drop(columns=[c for c in cols if c in df.columns])


def handle_returns(df, mode='net'):
    """'net': weekly units = sold - returned (clipped at 0 later). 'ignore': returns count as 0."""
    if mode == 'net':
        return df
    if mode == 'ignore':
        return df.assign(units=df['units'].where(df['is_return'] != 1, 0.0))
    raise ValueError(mode)


def filter_sales_types(df, exclude=('CLEARANCE',)):
    excl = set(exclude) | {f'{t}_RETURN' for t in exclude}
    return df.assign(units=df['units'].where(~df['sales_type'].isin(excl), 0.0))

## 4. Daily → weekly, and the common panel format

- **`grid='rows'` (Set C)**: every *listed* day has a row, so a missing row means "not listed", not zero. Weeks with fewer than `min_listed_days` listed days get `units = NaN` (unknown).
- **`grid='calendar'` (Sets B, D)**: only transactions are stored, so every day from the series' first sale to the end of the data without a row is a 0.
- Partial first / last weeks of a dataset are dropped (a 2-day "week" would look like a sales drop).

**Panel format**: one row per `series_id × week_start` on a gap-free Monday grid. Columns: `source`, `series_id` (`source:id`), `week_start`, `units` (`NaN` = unknown), `censored`, `store_id`, `article_id`, optional hierarchy columns and `on_hand` (test shop only).

In [ ]:
def _weekly_grid(span, keys):
    n = ((span['max'] - span['min']).dt.days // 7 + 1).to_numpy()
    grid = pd.DataFrame(index=span.index.repeat(n)).reset_index()
    offset = grid.groupby(keys, sort=False).cumcount().to_numpy()
    grid['week_start'] = np.repeat(span['min'].to_numpy(), n) + pd.to_timedelta(offset * 7, unit='D')
    return grid


def daily_to_weekly(df, keys, grid='rows', min_listed_days=1, drop_partial_edge_weeks=True,
                    date_col='date', static_cols=()):
    d = df.assign(week_start=to_week(df[date_col]))
    first_day, last_day = d[date_col].min(), d[date_col].max()
    w = d.groupby([*keys, 'week_start'], observed=True).agg(
        units=('units', 'sum'), listed_days=(date_col, 'nunique')).reset_index()
    span = w.groupby(keys)['week_start'].agg(['min', 'max'])
    if grid == 'calendar':
        span['max'] = to_week(pd.Series([last_day])).iloc[0]
    out = _weekly_grid(span, keys).merge(w, on=[*keys, 'week_start'], how='left')
    if static_cols:
        out = out.merge(d.groupby(keys)[list(static_cols)].first().reset_index(), on=keys, how='left')
    if grid == 'rows':
        out['listed_days'] = out['listed_days'].fillna(0)
        out['units'] = out['units'].where(out['listed_days'] >= min_listed_days)
    elif grid == 'calendar':
        out['units'] = out['units'].fillna(0.0)
    else:
        raise ValueError(grid)
    if drop_partial_edge_weeks:
        out = out[(out['week_start'] >= first_day) & (out['week_start'] + pd.Timedelta(days=6) <= last_day)]
    return out.reset_index(drop=True)


def to_panel(df, source, id_cols, static_cols=(), extra_cols=()):
    """Standardise a weekly table to the panel format. extra_cols: numeric columns to keep (e.g. on_hand)."""
    sid = df[id_cols[0]].astype(str)
    for c in id_cols[1:]:
        sid = sid + '|' + df[c].astype(str)
    out = pd.DataFrame({
        'source': source,
        'series_id': source + ':' + sid,
        'week_start': df['week_start'].to_numpy(),
        'units': df['units'].astype(float).to_numpy(),
        'censored': df['censored'].fillna(False).astype(bool).to_numpy() if 'censored' in df else False,
        'store_id': df['store_id'].astype(str).to_numpy() if 'store_id' in df else None,
        'article_id': df['article_id'].astype(str).to_numpy(),
    })
    for c in static_cols:
        out[c] = df[c].astype(str).to_numpy()
    for c in extra_cols:
        out[c] = df[c].astype(float).to_numpy()
    return out.sort_values(['series_id', 'week_start'], ignore_index=True)

## 5. Panel cleaning steps (shared by all sources)

| Step | What it does |
|---|---|
| `clip_negative` | weekly net units < 0 (more returns than sales) → 0 |
| `mask_censored` | `units = NaN` where `censored` (neither a target nor a zero in the features) |
| `handle_outliers` | robust z-score per series: flag, cap or mask extreme weeks |
| `trim_to_first_sale` | drop the weeks before a series' first sale. **Not for the test shop** (see §8) |
| `drop_short_series` | drop series with too few weeks or too few sale weeks |
| `check_panel` | assert unique keys and a gap-free weekly grid |

In [ ]:
def clip_negative(panel, col='units'):
    return panel.assign(**{col: panel[col].clip(lower=0)})


def mask_censored(panel):
    return panel.assign(units=panel['units'].where(~panel['censored']))


def handle_outliers(panel, z=20, mode='flag', min_units=10):
    g = panel.groupby('series_id', sort=False)['units']
    med = g.transform('median')
    mad = (panel['units'] - med).abs().groupby(panel['series_id'], sort=False).transform('median').replace(0, 1)
    limit = med + z * 1.4826 * mad
    is_out = (panel['units'] > limit) & (panel['units'] >= min_units)
    if mode == 'flag':
        return panel.assign(is_outlier=is_out)
    if mode == 'cap':
        return panel.assign(units=panel['units'].mask(is_out, np.floor(limit)))
    if mode == 'nan':
        return panel.assign(units=panel['units'].mask(is_out))
    raise ValueError(mode)


def trim_to_first_sale(panel):
    started = (panel['units'].fillna(0) > 0).groupby(panel['series_id'], sort=False).cummax()
    return panel[started].reset_index(drop=True)


def drop_short_series(panel, min_weeks=26, min_sale_weeks=2):
    g = panel.groupby('series_id', sort=False)
    n_weeks = g['week_start'].transform('size')
    n_sales = (panel['units'] > 0).groupby(panel['series_id'], sort=False).transform('sum')
    return panel[(n_weeks >= min_weeks) & (n_sales >= min_sale_weeks)].reset_index(drop=True)


def check_panel(panel):
    assert not panel.duplicated(['series_id', 'week_start']).any(), 'duplicate series_id x week_start'
    assert panel['week_start'].dt.day_of_week.eq(0).all(), 'week_start must be a Monday'
    gaps = panel.groupby('series_id', sort=False)['week_start'].diff().dropna().dt.days
    assert gaps.eq(7).all(), 'weekly grid has gaps (or is not sorted)'
    return panel

## 6. Configure the sources ← edit here

Each source has a `load` function and a list of `steps`. Set `'enabled': False` to leave a source out of the panel entirely. (To keep a source in the panel but out of *training*, use `TRAIN_SOURCES` in §12 instead; §16 compares both.)

| Source | Level | Preparation |
|---|---|---|
| Test shop | article × week | flag possibly-censored weeks, keep `on_hand`. **Not** trimmed to the first sale |
| Set A | store × article × week | mask `in_stock = False` weeks; outliers only flagged |
| Set B | store × article × week | Monday weeks, zeros from the first sale on; at least 13 weeks (B has ~43) |
| Set C | store × article × week | drop leaky columns and the burst, net returns, only fully listed weeks |
| Set D | article × week (chain) | net returns, Monday weeks, from the first sale on |

In [ ]:
A_HIERARCHY = ['product_group', 'division', 'department', 'department_group', 'store_format']
CD_HIERARCHY = ['division', 'category', 'subcategory', 'segment']

COMMON_STEPS = [clip_negative, check_panel]
EXTRA_STEPS = [trim_to_first_sale, (drop_short_series, {'min_weeks': 26, 'min_sale_weeks': 2})]

SOURCES = {
    'test_shop': {
        'enabled': True,
        'load': load_test_shop,
        'steps': [
            (ts_flag_censored, {'max_on_hand': 0}),
            (to_panel, {'source': 'test_shop', 'id_cols': ['article_id'], 'extra_cols': ['on_hand']}),
            # mask_censored,                         # tested: doesn't help, on_hand is too noisy
            *COMMON_STEPS,
        ],
    },
    'A': {
        'enabled': True,
        'load': load_set_a,
        'steps': [
            a_flag_out_of_stock,
            (to_panel, {'source': 'A', 'id_cols': PAIR, 'static_cols': A_HIERARCHY}),
            mask_censored,
            (handle_outliers, {'z': 20, 'mode': 'flag'}),
            *COMMON_STEPS,
            *EXTRA_STEPS,
        ],
    },
    'B': {
        'enabled': True,
        'load': load_set_b,
        'steps': [
            (daily_to_weekly, {'keys': PAIR, 'grid': 'calendar', 'static_cols': ['category', 'subcategory']}),
            (to_panel, {'source': 'B', 'id_cols': PAIR, 'static_cols': ['category', 'subcategory']}),
            *COMMON_STEPS,
            trim_to_first_sale,
            (drop_short_series, {'min_weeks': 13, 'min_sale_weeks': 2}),   # B has < 1 year of history
        ],
    },
    'C': {
        'enabled': True,
        'load': load_set_c,
        'steps': [
            (drop_pairs, {'pairs': C_OUTLIER_PAIRS}),
            (drop_columns, {'cols': C_LEAKY_COLS}),
            (handle_returns, {'mode': 'net'}),
            (daily_to_weekly, {'keys': PAIR, 'grid': 'rows', 'min_listed_days': 7, 'static_cols': CD_HIERARCHY[1:]}),
            (to_panel, {'source': 'C', 'id_cols': PAIR, 'static_cols': CD_HIERARCHY[1:]}),
            *COMMON_STEPS,
            *EXTRA_STEPS,
        ],
    },
    'D': {
        'enabled': True,
        'load': load_set_d,
        'steps': [
            (drop_columns, {'cols': D_CONSTANT_COLS}),
            (handle_returns, {'mode': 'net'}),
            (daily_to_weekly, {'keys': ['article_id'], 'grid': 'calendar', 'static_cols': ['supplier_id', *CD_HIERARCHY]}),
            (to_panel, {'source': 'D', 'id_cols': ['article_id'], 'static_cols': ['supplier_id', *CD_HIERARCHY]}),
            *COMMON_STEPS,
            trim_to_first_sale,
            (drop_short_series, {'min_weeks': 8, 'min_sale_weeks': 2}),    # D has < 1 year of history
        ],
    },
}


def build_panel(sources=SOURCES, verbose=True):
    panels = []
    for name, cfg in sources.items():
        if not cfg.get('enabled', True):
            continue
        raw = cfg['load']()
        panels.append(run_pipeline(raw, cfg['steps'], name=name, verbose=verbose))
    panel = check_panel(pd.concat(panels, ignore_index=True))
    panel['source'] = panel['source'].astype('category')
    return panel


def sb_label(adi, cv2):
    """Syntetos-Boylan demand class from ADI (avg. weeks between sales) and CV² of the non-zero sizes."""
    return np.select([(adi < 1.32) & (cv2 < 0.49), (adi >= 1.32) & (cv2 < 0.49), adi < 1.32],
                     ['smooth', 'intermittent', 'erratic'], 'lumpy')


def series_summary(panel):
    """Per source: size, unknown / zero weeks and demand classes (counted from each series' first sale)."""
    p = trim_to_first_sale(panel)
    u = p['units']
    g = p.assign(obs=u.notna(), nz=u > 0, nzu=u.where(u > 0)).groupby(['source', SID], observed=True)
    st = g.agg(weeks=('week_start', 'size'), obs=('obs', 'sum'), nz=('nz', 'sum'), mean_nz=('nzu', 'mean'), std_nz=('nzu', 'std'))
    st['class'] = sb_label(st.obs / st.nz, (st.std_nz / st.mean_nz).pow(2).fillna(0))
    rows = {}
    for src, s in st.groupby(level='source', observed=True):
        rows[src] = {'series': len(s), 'weeks (median)': s.weeks.median(),
                     'unknown_weeks_%': round(100 * (1 - s.obs.sum() / s.weeks.sum()), 1),
                     'zero_weeks_%': round(100 * (1 - s.nz.sum() / s.obs.sum()), 1),
                     'units per sale week (median)': round(s.mean_nz.median(), 2),
                     **{f'{c}_%': round(100 * (s['class'] == c).mean(), 1) for c in ['smooth', 'intermittent', 'erratic', 'lumpy']}}
    return pd.DataFrame(rows).T


panel = build_panel()
print(f'\nPanel: {len(panel):,} rows, {panel[SID].nunique():,} series')
display(series_summary(panel))

The summary shows why DATASETS.md called Set B "too smooth": about a quarter of its weeks are zeros, against ~60% in the test shop, and most of its series are smooth. Whether it helps anyway is measured in §16.

## 7. Feature functions

Every row of the panel becomes a **forecast origin** `t` (the last known week). Features use only weeks up to `t`, targets are `t+1 … t+13`. `lag_1` is the origin week itself.

| Function | Adds | Notes |
|---|---|---|
| `add_scale` | `scale` | the series' mean week-to-week change up to `t` (the MASE denominator, expanding so leak-free) |
| `add_lags` | `lag_k` | |
| `add_rolling` | `roll_{stat}_{w}` | mean, median, quantiles, max |
| `add_zero_share` | `zero_share_{w}` | share of zero weeks |
| `add_recency` | `weeks_since_sale`, `age_weeks` | dead / new article signals |
| `add_intermittency` | `adi_52`, `cv2_52`, `demand_class` | rolling Syntetos–Boylan class |
| `add_seasonal` | `seasonal_h{h}`, `seas_ratio_52` | same weeks last year (lag-based; the test shop's dates are shifted) |
| **`add_launch_features`** | `launch_mean`, `launch_first`, `launch_max`, `sale_share_since_launch`, `n_sale_weeks` | sales **since the first sale**: for new articles, rolling windows are diluted by pre-launch zeros |
| **`add_stock_features`** | `oh_level`, `oh_change`, `stock_out`, `stock_log`, `stock_cover` | from `on_hand` (test shop only, `NaN` elsewhere). `stock_cover` = stock ÷ 13-week mean sales |
| `add_targets` | `target_h{h}` | units in week `t+h` |
| `add_source_info` | `is_test_shop`, `sample_weight` | test shop weighted ×3 |
| `apply_scaling` | | divides lags, rolling stats, seasonal, launch, stock level and targets by `scale` |

In [ ]:
def _grp(panel, col):
    return panel.groupby(panel[SID], sort=False, observed=True)[col] if isinstance(col, str) else col.groupby(panel[SID], sort=False)


def _roll(panel, s, window, how='mean', **kw):
    g = s.groupby(panel[SID], sort=False)
    r = g.expanding(min_periods=1) if window is None else g.rolling(window, min_periods=1)
    res = r.quantile(kw['q']) if how == 'quantile' else getattr(r, how)()
    return res.reset_index(level=0, drop=True).sort_index()


def add_scale(panel, method='mase', window=None, col='units'):
    u = panel[col]
    base = {'mase': _grp(panel, col).diff().abs(), 'mean': u, 'nonzero_mean': u.where(u > 0)}[method]
    scale = _roll(panel, base, window)
    fallback = _roll(panel, u.where(u > 0), window)
    scale = scale.where(scale > 0, fallback)
    return panel.assign(scale=scale.where(scale > 0, 1.0).fillna(1.0))


def add_lags(panel, lags=(*range(1, 14), 26, 52), col='units'):
    g = _grp(panel, col)
    return panel.assign(**{f'lag_{k}': g.shift(k - 1) for k in lags})


def add_rolling(panel, windows=(4, 8, 13, 26, 52), stats=('mean', 'median'), col='units'):
    new = {}
    for w in windows:
        for st in stats:
            new[f'roll_{st}_{w}'] = (_roll(panel, panel[col], w, 'quantile', q=int(st[1:]) / 100)
                                     if st.startswith('q') else _roll(panel, panel[col], w, st))
    return panel.assign(**new)


def add_zero_share(panel, windows=(13, 52), col='units'):
    is_zero = (panel[col] == 0).astype(float).where(panel[col].notna())
    return panel.assign(**{f'zero_share_{w}': _roll(panel, is_zero, w) for w in windows})


def add_recency(panel, col='units'):
    sold = panel[col].fillna(0) > 0
    pos = panel.groupby(SID, sort=False, observed=True).cumcount().astype(float)
    last_sale = _grp(panel, pos.where(sold)).ffill()
    started = _grp(panel, sold).cummax()
    age = _grp(panel, started.astype(int)).cumsum() - 1
    return panel.assign(weeks_since_sale=pos - last_sale, age_weeks=age.where(started))


def add_intermittency(panel, window=52, col='units'):
    u = panel[col]
    nz = (u > 0).astype(float)
    x = u.where(u > 0, 0).fillna(0)
    n_nz, n_obs = _roll(panel, nz, window, 'sum'), _roll(panel, u.notna().astype(float), window, 'sum')
    s1, s2 = _roll(panel, x, window, 'sum'), _roll(panel, x ** 2, window, 'sum')
    adi = (n_obs / n_nz).where(n_nz > 0)
    mean = s1 / n_nz
    var = ((s2 - n_nz * mean ** 2) / (n_nz - 1)).where(n_nz > 1, 0).clip(lower=0)
    cv2 = (var / mean ** 2).where(n_nz > 0)
    cls = pd.Series(sb_label(adi.fillna(np.inf), cv2.fillna(0)), index=panel.index).where(n_nz > 0, 'no_sales')
    return panel.assign(**{f'adi_{window}': adi, f'cv2_{window}': cv2,
                           'demand_class': pd.Categorical(cls, ['smooth', 'intermittent', 'erratic', 'lumpy', 'no_sales'])})


def add_seasonal(panel, horizon=H, col='units', ratio=True, smoothing=1.0):
    g = _grp(panel, col)
    new = {f'seasonal_h{h}': g.shift(52 - h) for h in range(1, horizon + 1)}
    if ratio:
        r13 = _roll(panel, panel[col], 13)
        new['seas_ratio_52'] = (_grp(panel, r13).shift(52 - 13) + smoothing) / (_grp(panel, r13).shift(52) + smoothing)
    return panel.assign(**new)


def add_launch_features(panel, col='units'):
    """Sales since the series' first sale. NaN before the first sale."""
    u = panel[col].fillna(0)
    sold = u > 0
    started = _grp(panel, sold).cummax()
    us = u.where(started, 0.0)
    n = _grp(panel, started.astype(float)).cumsum()            # weeks since the first sale (incl.)
    nsale = _grp(panel, sold.astype(float)).cumsum()
    return panel.assign(
        launch_mean=(_grp(panel, us).cumsum() / n).where(started),
        launch_first=_grp(panel, u.where(sold & (nsale == 1))).ffill(),
        launch_max=_grp(panel, us).cummax().where(started),
        sale_share_since_launch=(nsale / n).where(started),
        n_sale_weeks=nsale)


def add_stock_features(panel, col='on_hand'):
    """Features from end-of-week stock. Sources without stock data get NaN (LightGBM handles it)."""
    if col not in panel:
        return panel
    oh = panel[col]
    rm = panel['roll_mean_13'] if 'roll_mean_13' in panel else _roll(panel, panel['units'], 13)
    return panel.assign(oh_level=oh, oh_change=oh - _grp(panel, oh).shift(1),
                        stock_out=(oh <= 0).astype(float).where(oh.notna()),
                        stock_log=np.log1p(oh.clip(lower=0)),
                        stock_cover=(oh.clip(lower=0) / rm.clip(lower=0.1)).where(oh.notna()))


def add_targets(panel, horizon=H, col='units'):
    g = _grp(panel, col)
    return panel.assign(**{f'target_h{h}': g.shift(-h) for h in range(1, horizon + 1)})


def add_source_info(panel, weights=None):
    w = panel['source'].astype(str).map(weights or {}).fillna(1.0).astype(float)
    return panel.assign(is_test_shop=(panel['source'] == 'test_shop').astype(int), sample_weight=w)


SCALED_PREFIXES = ('lag_', 'roll_', 'seasonal_h', 'target_h', 'launch_', 'oh_')


def apply_scaling(panel, prefixes=SCALED_PREFIXES, scale_col='scale'):
    cols = [c for c in panel.columns if c.startswith(prefixes)]
    out = panel.copy()
    out[cols] = out[cols].div(out[scale_col], axis=0)
    out['log_scale'] = np.log(out[scale_col])
    return out


def downcast(panel):
    return panel.astype({c: 'float32' for c in panel.select_dtypes('float64').columns})

## 8. Configure the features ← edit here, then build the frame

Order matters only for `add_scale` → `apply_scaling` and for `add_stock_features` after `add_rolling` (it uses `roll_mean_13`).

In [ ]:
FEATURE_STEPS = [
    (add_scale, {'method': 'mase', 'window': None}),
    (add_lags, {'lags': (*range(1, 14), 26, 52)}),
    (add_rolling, {'windows': (4, 8, 13, 26, 52), 'stats': ('mean', 'median')}),
    (add_rolling, {'windows': (13, 26), 'stats': ('q25', 'q75', 'max')}),
    (add_zero_share, {'windows': (13, 52)}),
    add_recency,
    (add_intermittency, {'window': 52}),
    (add_seasonal, {'horizon': H, 'ratio': True}),
    add_launch_features,
    add_stock_features,
    (add_targets, {'horizon': H}),
    (add_source_info, {'weights': {'test_shop': 3.0}}),
    apply_scaling,
    downcast,
]

frame = run_pipeline(panel, FEATURE_STEPS, name='features')
print(f'feature frame: {len(frame):,} rows x {frame.shape[1]} cols')

# Raw test-shop matrix (articles x weeks): every score is computed on it, never on the feature frame
ts_raw = load_test_shop()
Y = ts_raw.pivot(index='article_id', columns='week_start', values='units').sort_index(axis=1)
print(f'test shop: {Y.shape[0]:,} articles x {Y.shape[1]} weeks ({Y.columns[0].date()} -> {Y.columns[-1].date()})')

## 9. Train / validation sets and the metric

- **Cutoff** = the last known week of a backtest. Training uses every origin of the extra sets and the test-shop origins whose 13 target weeks end by the cutoff. Validation = the test shop's 2,904 articles at the cutoff.
- `to_long` turns one row per origin (`target_h1..13`) into one row per origin × horizon (`h`, `target`), which is what one direct model trains on.
- `subsample_origins` keeps every k-th origin per source (`ORIGIN_STRIDE`), plus **every** origin of a series younger than `young_weeks` (launches are rare and expensive in the metric).
- `mase_score` is the competition metric; `segment_points` splits the score by article age at the cutoff. The points of all segments add up to the total score.

**Why the test shop is not trimmed to its first sale:** the metric's denominator includes the zeros before the launch, and the features must see the same history. Trimming made the median baseline ~20 points worse.

In [ ]:
META_COLS = ['source', SID, 'week_start', 'units', 'censored', 'store_id', 'article_id', 'scale', 'is_outlier',
             'sample_weight', 'on_hand', 'supplier_id', *A_HIERARCHY, *CD_HIERARCHY]
LONG_ID_COLS = ['source', SID, 'article_id', 'week_start', 'scale', 'sample_weight']
CAT_FEATURES = ['demand_class']


def rolling_cutoffs(frame, n_windows=4, step=H, horizon=H, source='test_shop'):
    """Latest first: the last week with a full 13-week holdout, then every `step` weeks back."""
    last = frame.loc[frame['source'] == source, 'week_start'].max()
    return [last - pd.Timedelta(weeks=horizon + i * step) for i in range(n_windows)]


def split_frame(frame, cutoff=None, horizon=H, valid_source='test_shop'):
    is_v = frame['source'] == valid_source
    if cutoff is None:
        cutoff = frame.loc[is_v, 'week_start'].max()
    target_end = frame['week_start'] + pd.Timedelta(weeks=horizon)
    return (frame[~is_v | (target_end <= cutoff)].reset_index(drop=True),
            frame[is_v & (frame['week_start'] == cutoff)].reset_index(drop=True))


def subsample_origins(frame, every=None, young_weeks=0, min_history_weeks=1):
    k = frame['source'].astype(str).map(every or {}).fillna(1).astype(int)
    from_end = frame.groupby(SID, sort=False, observed=True).cumcount(ascending=False)
    keep = (from_end % k == 0)
    if young_weeks:
        keep |= frame['age_weeks'] < young_weeks
    keep &= frame['age_weeks'] >= min_history_weeks - 1          # drop origins before the first sale
    return frame[keep].reset_index(drop=True)


def to_long(frame, horizon=H, drop_missing_target=True, id_cols=LONG_ID_COLS):
    feats = [c for c in frame.columns if c not in META_COLS and not c.startswith(('target_h', 'seasonal_h'))]
    base = [c for c in id_cols if c in frame.columns] + feats
    parts = []
    for h in range(1, horizon + 1):
        p = frame[base].copy()
        p['h'] = np.int8(h)
        if f'seasonal_h{h}' in frame:
            p['seasonal_lag52'] = frame[f'seasonal_h{h}'].to_numpy()
        if f'target_h{h}' in frame:
            p['target'] = frame[f'target_h{h}'].to_numpy()
        parts.append(p)
    long = pd.concat(parts, ignore_index=True)
    long[[SID, 'article_id']] = long[[SID, 'article_id']].astype('category')
    if drop_missing_target and 'target' in long:
        long = long[long['target'].notna()].reset_index(drop=True)
    return long


def forecast_matrix(rows, pred, index=None, horizon=H):
    """Long rows (article_id, h) + predictions -> articles x horizon array (order of Y.index); missing -> 0."""
    index = Y.index if index is None else index
    F = pd.DataFrame({'article_id': rows['article_id'].astype(str).to_numpy(), 'h': rows['h'].to_numpy(),
                      'pred': np.asarray(pred, dtype=float)}).pivot(index='article_id', columns='h', values='pred')
    return F.reindex(index=index, columns=range(1, horizon + 1)).fillna(0).to_numpy(float)


def per_article_mase(cutoff, F, horizon=H):
    """MASE per article; NaN for articles with no movement before the cutoff (they can't be scored)."""
    hist = Y.loc[:, :cutoff]
    act = Y.loc[:, Y.columns > cutoff].iloc[:, :horizon].to_numpy()
    den = hist.diff(axis=1).abs().mean(axis=1).to_numpy()
    err = np.abs(act - np.clip(F, 0, None))
    with np.errstate(divide='ignore', invalid='ignore'):
        return {name: np.where(den > 0, err[:, cols].mean(axis=1) / den, np.nan)
                for name, cols in [('all', slice(0, horizon)), ('w1-w7', slice(0, 7)), ('w8-w13', slice(7, horizon))]}


def mase_score(cutoff, F):
    """Competition score (1 - MASE) x 100, higher is better."""
    return {k: round((1 - np.nanmean(v)) * 100, 2) for k, v in per_article_mase(cutoff, F).items()}


AGE_SEGMENTS = ['launched 0-3 wk ago', 'launched 4-12 wk ago', 'launched 13-51 wk ago', 'older']


def article_segments(cutoff):
    """Article age at the cutoff, counted from the first sale."""
    hist = Y.loc[:, :cutoff].to_numpy()
    started = hist > 0
    first = np.where(started.any(1), started.argmax(1), -1)
    age = hist.shape[1] - 1 - first
    return np.select([first < 0, age < 4, age < 13, age < 52], ['never sold', *AGE_SEGMENTS[:3]], AGE_SEGMENTS[3])


def segment_points(cutoff, F):
    """Each segment's share of the total score (in points). The segments add up to the score."""
    m = per_article_mase(cutoff, F)['all']
    seg, ok = article_segments(cutoff), np.isfinite(m)
    return {s: ((1 - m[ok & (seg == s)]).sum()) / ok.sum() * 100 for s in AGE_SEGMENTS}


def make_submission(F):
    F = np.clip(np.asarray(F, dtype=float), 0, None)
    sub = pd.DataFrame(F, index=Y.index, columns=[f'w{i}' for i in range(1, H + 1)]).reset_index()
    assert len(sub) == 2904 and sub.article_id.is_unique and not sub.isna().any().any()
    return sub


# Sanity check: "median of the last 13 weeks" from the feature frame must equal the EDA baseline
# (-6.7, -30.5, -12.6, -16.3 on the four windows) if lags, scale and targets are aligned.
for c in rolling_cutoffs(frame):
    v = to_long(split_frame(frame, c)[1], drop_missing_target=False)
    from_frame = mase_score(c, forecast_matrix(v, v['roll_median_13'] * v['scale']))['all']
    eda = mase_score(c, np.repeat(Y.loc[:, :c].iloc[:, -13:].median(axis=1).to_numpy()[:, None], H, axis=1))['all']
    print(f'cutoff {c.date()}: median baseline from the frame {from_frame:7.2f} | from the raw history {eda:7.2f}')
    assert abs(from_frame - eda) < 0.01, 'feature frame is misaligned'

## 10. Models

Each `fit_*` returns `(F, fitted)`: `F` is the articles × 13 forecast for the context's cutoff, and `fitted.predict(other_ctx)` applies the same fitted model to another cutoff (used for the overfitting check).

- **References** (no training): `median_13w` (the baseline to beat), `median_26w`, `naive_last` (last week repeated) and `seasonal_naive_52` (same weeks last year).
- **LightGBM**: L1 loss on the scaled target (= the metric's weighting: error ÷ the series' typical weekly change).
- **Linear L1**: `SGDRegressor` with absolute loss, inputs log-transformed, imputed and standardised.
- **MLP**: squared loss on `log1p(target)`, small predictions snapped to 0.
- **TSB**: per-article smoothing of sale probability and size; forecast the size if the probability > 0.5, else 0 (the median version).

In [ ]:
class Fitted:
    def __init__(self, predict, model=None):
        self.predict, self.model = predict, model


def _fitted(ctx, predict, model=None):
    return predict(ctx), Fitted(predict, model)


def _repeat(f):
    return np.repeat(np.asarray(f, dtype=float)[:, None], H, axis=1)


def make_context(frame, cutoff=None, origin_stride=None, young_weeks=0, train_sources=None, with_train=True):
    """Everything a model needs for one cutoff, built once so all models see the same data."""
    train_w, valid_w = split_frame(frame, cutoff)
    if train_sources is not None:
        train_w = train_w[train_w['source'].astype(str).isin(train_sources)]
    cutoff = valid_w['week_start'].max()
    valid = to_long(valid_w, drop_missing_target=False)
    at_cut = valid_w.assign(article_id=valid_w['article_id'].astype(str)).set_index('article_id').reindex(Y.index)
    return {
        'cutoff': cutoff,
        'train': to_long(subsample_origins(train_w, origin_stride, young_weeks)) if with_train else None,
        'valid': valid,
        'features': [c for c in valid.columns if c not in LONG_ID_COLS and c != 'target'],
        'Y_hist': Y.loc[:, :cutoff],
        'weeks_since_sale': at_cut['weeks_since_sale'].to_numpy(float),
        'demand_class': at_cut['demand_class'].astype(str).to_numpy(),
    }


# ---------- reference models ----------
def fit_median(ctx, weeks=13):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -weeks:].median(axis=1)))


def fit_naive(ctx):
    return _fitted(ctx, lambda c: _repeat(c['Y_hist'].iloc[:, -1]))


def fit_seasonal_naive(ctx, season=52):
    return _fitted(ctx, lambda c: c['Y_hist'].iloc[:, -season:-season + H].to_numpy(float))


# ---------- global models ----------
def _global_predictor(model, feats, transform=None):
    def predict(c):
        va = c['valid']
        out = model.predict(va[feats])
        if transform is not None:
            out = transform(out)
        return forecast_matrix(va, np.clip(out, 0, None) * va['scale'].to_numpy())
    return predict


LGB_PARAMS = dict(objective='l1', learning_rate=0.1, num_leaves=63, min_data_in_leaf=1000, max_bin=63,
                  feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=1.0,
                  force_col_wise=True, verbose=-1, seed=SEED)


def fit_lightgbm(ctx, params=LGB_PARAMS, rounds=200, use_weights=True):
    tr, feats = ctx['train'], ctx['features']
    ds = lgb.Dataset(tr[feats], tr['target'], weight=tr['sample_weight'] if use_weights else None,
                     categorical_feature=[c for c in CAT_FEATURES if c in feats])
    model = lgb.train(params, ds, num_boost_round=rounds)
    return _fitted(ctx, _global_predictor(model, feats), model)


def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))


def make_tabular_preprocessor(features, categorical=(*CAT_FEATURES, 'h')):
    cat = [c for c in categorical if c in features]
    num = [c for c in features if c not in cat]
    return ColumnTransformer([
        ('num', make_pipeline(FunctionTransformer(signed_log), SimpleImputer(strategy='median', add_indicator=True),
                              StandardScaler()), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
    ])


def _sample(df, n, seed=SEED):
    return df.sample(n=n, random_state=seed) if n and len(df) > n else df


def fit_linear_l1(ctx, sample=1_000_000, alpha=1e-5, epochs=10, target_clip_q=0.999, use_weights=True):
    tr, feats = _sample(ctx['train'], sample), ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          SGDRegressor(loss='epsilon_insensitive', epsilon=0.0, alpha=alpha, average=True,
                                       max_iter=epochs, tol=None, random_state=SEED))
    y = tr['target'].clip(upper=tr['target'].quantile(target_clip_q))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], y, sgdregressor__sample_weight=tr['sample_weight'].to_numpy() if use_weights else None)
    return _fitted(ctx, _global_predictor(model, feats), model)


def fit_mlp(ctx, sample=500_000, hidden=(128, 64), max_iter=30, alpha=1e-4, zero_below=0.5):
    tr, feats = _sample(ctx['train'], sample), ctx['features']
    model = make_pipeline(make_tabular_preprocessor(feats),
                          MLPRegressor(hidden_layer_sizes=hidden, batch_size=2048, learning_rate_init=1e-3, alpha=alpha,
                                       max_iter=max_iter, early_stopping=True, n_iter_no_change=4, random_state=SEED))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        model.fit(tr[feats], np.log1p(tr['target'].clip(lower=0)))

    def back(out):
        out = np.expm1(out).clip(0)
        return np.where(out < zero_below, 0, out)
    return _fitted(ctx, _global_predictor(model, feats, back), model)


# ---------- young-article model ----------
YOUNG_PARAMS = dict(objective='l1', learning_rate=0.05, num_leaves=15, min_data_in_leaf=50, feature_fraction=0.8,
                    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, seed=SEED)


def _young_rows(df, max_age):
    return df[(df['source'] == 'test_shop') & (df['age_weeks'] >= 0) & (df['age_weeks'] < max_age)]


def _level(df):
    """The article's sales level since launch, in units (launch_mean is stored divided by scale)."""
    return (df['launch_mean'] * df['scale']).clip(lower=0.25).to_numpy()


def fit_young_lgb(ctx, max_age=13, params=YOUNG_PARAMS, rounds=300):
    """LightGBM for test-shop articles launched less than `max_age` weeks ago. Other articles get NaN,
    so it is only used through the `override` ensemble (§12).

    The main model's target is units / scale. For a new article the scale is ~0.01, so forecasting 2 units
    means predicting ~200, and no tree leaf gets there: the main model forecasts 0 for every new article.
    Here the target is units / level (values around 1) and the weight is level / scale. The loss is the
    same as the metric's, |y - forecast| / scale, but the numbers are well-behaved."""
    tr, feats = _young_rows(ctx['train'], max_age), ctx['features']
    level = _level(tr)
    y = tr['target'].to_numpy() * tr['scale'].to_numpy() / level
    ds = lgb.Dataset(tr[feats], y, weight=level / tr['scale'].to_numpy(),
                     categorical_feature=[c for c in CAT_FEATURES if c in feats])
    model = lgb.train(params, ds, num_boost_round=rounds)

    def predict(c):
        va = c['valid']
        is_young = ((va['age_weeks'] >= 0) & (va['age_weeks'] < max_age)).to_numpy()
        pred = np.full(len(va), np.nan)
        if is_young.any():
            pred[is_young] = np.clip(model.predict(va.loc[is_young, feats]), 0, None) * _level(va[is_young])
        F = pd.DataFrame({'article_id': va['article_id'].astype(str).to_numpy(), 'h': va['h'].to_numpy(),
                          'pred': pred}).pivot(index='article_id', columns='h', values='pred')
        return F.reindex(index=Y.index, columns=range(1, H + 1)).to_numpy(float)   # NaN = not young
    return _fitted(ctx, predict, model)


def override(stack, axis=0):
    """Ensemble rule: the second member's forecast where it has one (not NaN), else the first member's."""
    return np.where(np.isnan(stack[1]), stack[0], stack[1])


# ---------- TSB ----------
def tsb_state(Y_hist, alpha=0.1, beta=0.05):
    X = Y_hist.to_numpy(float)
    p, z = np.zeros(len(X)), np.full(len(X), np.nan)
    for t in range(X.shape[1]):
        d = X[:, t]
        sale = d > 0
        z = np.where(sale, np.where(np.isnan(z), d, z + alpha * (d - z)), z)
        p = p + beta * (sale - p)
    return p, np.nan_to_num(z)


def fit_tsb(ctx, alpha=0.1, beta=0.05, threshold=0.5):
    def predict(c):
        p, z = tsb_state(c['Y_hist'], alpha, beta)
        return _repeat(np.where(p > threshold, z, 0.0))
    return _fitted(ctx, predict)

## 11. Post-processing

Applied to every forecast after clipping at 0, in the order of `POSTPROCESS` (§12).

- `dead_to_zero`: forecast 0 for articles with no sale in the last 8 weeks (the site's baselines do the same).
- `snap_small`: forecasts below 0.5 → 0 (the median of a mostly-zero week is 0).
- `round_to_int`: whole units. Sales are whole numbers and the best forecast under absolute error is a median, which is a whole number too. It improved every learner on all 4 windows (by ~0.3 points).

In [ ]:
def dead_to_zero(F, ctx, weeks=8):
    dead = ~(ctx['weeks_since_sale'] < weeks)          # NaN (never sold) counts as dead
    return np.where(dead[:, None], 0.0, F)


def snap_small(F, ctx, below=0.5):
    return np.where(F < below, 0.0, F)


def round_to_int(F, ctx):
    return np.round(F)


def postprocess(F, ctx, steps):
    F = np.clip(F, 0, None)
    for step in steps:
        func, kw = (step, {}) if callable(step) else step
        F = func(F, ctx, **kw)
    return F

## 12. Configure the models ← edit here

- `MODELS`: name → `(fit function, options)`. Comment out a line to skip a model.
- `REFERENCES`: comparison-only models, never chosen as the final model.
- `COMPONENTS`: models that forecast only some articles (the young-article model); they are used inside ensembles only.
- `ENSEMBLES`: name → `(combine function, members)`, built from the members' raw forecasts. `override` takes the second member's forecast where it has one.
- `TRAIN_SOURCES`: which sources the global models learn from (the panel can hold more).
- `ORIGIN_STRIDE`: keep every k-th origin per source; `YOUNG_WEEKS`: keep every origin younger than this.

In [ ]:
MODELS = {
    'median_13w':        (fit_median, {'weeks': 13}),
    'median_26w':        (fit_median, {'weeks': 26}),
    'naive_last':        (fit_naive, {}),
    'seasonal_naive_52': (fit_seasonal_naive, {}),
    'lightgbm':          (fit_lightgbm, {'rounds': 200}),
    'linear_l1':         (fit_linear_l1, {'sample': 1_000_000}),
    'mlp':               (fit_mlp, {'sample': 500_000, 'hidden': (128, 64)}),
    'tsb':               (fit_tsb, {'alpha': 0.1, 'beta': 0.05}),
    'young_lgb':         (fit_young_lgb, {'max_age': 13}),
}
REFERENCES = ['median_13w', 'median_26w', 'naive_last', 'seasonal_naive_52']
COMPONENTS = ['young_lgb']        # only forecast some articles: used inside ensembles, not scored alone
BASELINE = 'median_13w'

ENSEMBLES = {
    'lightgbm + young model': (override, ['lightgbm', 'young_lgb']),
    'ens_mean(lgb+linear)': (np.mean, ['lightgbm', 'linear_l1']),
    'ens_median(4 models)': (np.median, ['lightgbm', 'linear_l1', 'mlp', 'tsb']),
}

POSTPROCESS = [
    (dead_to_zero, {'weeks': 8}),
    (snap_small, {'below': 0.5}),
    round_to_int,
]

TRAIN_SOURCES = ['test_shop', 'A', 'C', 'D']      # add 'B' to train on Set B too (§16: slightly worse)
ORIGIN_STRIDE = {'test_shop': 2, 'A': 4, 'B': 4, 'C': 4, 'D': 2}
YOUNG_WEEKS = 13                   # every origin < 13 weeks after launch, for the young-article model
N_WINDOWS = 4
TUNING_WINDOWS = 1

SITE_SCORES = {'AutoARIMA (site)': -33.7, 'naive (site)': -64.5}   # measured on the real hidden w1-w7

## 13. Rolling backtest

**No random split**: shuffling weeks would let a model learn from the future. For each of `N_WINDOWS` cutoffs, 13 weeks apart, the models train only on data up to the cutoff and are scored on the 13 weeks after it, exactly like the real task.

- The **most recent window** is the *tuning* window: settings were chosen by looking at it. The older windows are the **test windows**.
- **In-sample check**: each fitted model is also scored on the 13 weeks *before* its cutoff (its own training targets), for §15.
- Scores move a lot between windows (−5 to −30 for the same model), so always compare models window by window.

In [ ]:
def combine(store, members, how, cutoff):
    return how(np.stack([store[(m, cutoff)] for m in members]), axis=0)


def _score_rows(name, cutoff, F, F_in, ctx, ctx_in, secs):
    rows = []
    for variant, G, c in [('raw', np.clip(F, 0, None), ctx), ('post', postprocess(F, ctx, POSTPROCESS), ctx),
                          ('in-sample', postprocess(F_in, ctx_in, POSTPROCESS), ctx_in)]:
        rows.append({'model': name, 'cutoff': cutoff.date(), 'variant': variant, 'fit_s': round(secs, 1),
                     **mase_score(c['cutoff'], G)})
    return rows


def backtest(frame, models, ensembles, cutoffs):
    rows, forecasts, forecasts_in, fitted, contexts = [], {}, {}, {}, {}
    for i, cutoff in enumerate(cutoffs):
        t0 = time.perf_counter()
        ctx = make_context(frame, cutoff, ORIGIN_STRIDE, YOUNG_WEEKS, TRAIN_SOURCES)
        ctx_in = make_context(frame, cutoff - pd.Timedelta(weeks=H), with_train=False)
        contexts[cutoff] = {k: v for k, v in ctx.items() if k not in ('train', 'valid')}
        print(f'\n=== cutoff {cutoff.date()} | train {len(ctx["train"]):,} rows | context {time.perf_counter() - t0:.0f}s')
        print(f'  {"model":<22s} {"in-sample":>9s} {"raw":>8s} {"post":>8s}')
        for name, (fit, kw) in models.items():
            t0 = time.perf_counter()
            F, model = fit(ctx, **kw)
            secs = time.perf_counter() - t0
            forecasts[(name, cutoff)], forecasts_in[(name, cutoff)] = F, model.predict(ctx_in)
            if i == 0:
                fitted[name] = model
            if name in COMPONENTS:
                print(f'  {name:<22s} (component, used in ensembles)   ({secs:.0f}s)')
                continue
            new = _score_rows(name, cutoff, F, forecasts_in[(name, cutoff)], ctx, ctx_in, secs)
            rows += new
            print(f'  {name:<22s} {new[2]["all"]:9.2f} {new[0]["all"]:8.2f} {new[1]["all"]:8.2f}   ({secs:.0f}s)')
        for name, (how, members) in ensembles.items():
            if all((m, cutoff) in forecasts for m in members):
                forecasts[(name, cutoff)] = combine(forecasts, members, how, cutoff)
                forecasts_in[(name, cutoff)] = combine(forecasts_in, members, how, cutoff)
                new = _score_rows(name, cutoff, forecasts[(name, cutoff)], forecasts_in[(name, cutoff)], ctx, ctx_in, 0.0)
                rows += new
                print(f'  {name:<22s} {new[2]["all"]:9.2f} {new[0]["all"]:8.2f} {new[1]["all"]:8.2f}')
        del ctx
    return pd.DataFrame(rows), forecasts, forecasts_in, fitted, contexts


cutoffs = rolling_cutoffs(frame, n_windows=N_WINDOWS)
tuning_cutoffs, test_cutoffs = cutoffs[:TUNING_WINDOWS], cutoffs[TUNING_WINDOWS:]
print('Tuning window(s):', [c.date() for c in tuning_cutoffs], '| test windows:', [c.date() for c in test_cutoffs])
results, forecasts, forecasts_in, fitted, contexts = backtest(frame, MODELS, ENSEMBLES, cutoffs)

### Results

Score = (1 − MASE) × 100, **higher is better**. Post-processed. `w1-w7` is the live leaderboard, **`w8-w13` the final ranking**.

In [ ]:
post = results[results.variant == 'post']


def window_mean(df, cuts, cols=('all', 'w1-w7', 'w8-w13')):
    return df[df.cutoff.isin([c.date() for c in cuts])].groupby('model')[list(cols)].mean()


summary = pd.concat({'tuning window': window_mean(post, tuning_cutoffs),
                     'test windows': window_mean(post, test_cutoffs),
                     'all windows': window_mean(post, cutoffs)}, axis=1).round(2)
per_window = post.pivot(index='model', columns='cutoff', values='all')
summary[('', 'beats baseline on # windows')] = (per_window.gt(per_window.loc[BASELINE], axis=1)).sum(axis=1)
summary = summary.sort_values(('all windows', 'all'), ascending=False)
display(summary)
print('Per window (all w1-w13):')
display(per_window.loc[summary.index])

## 14. Where the score is won and lost: article age

Each cell is a segment's contribution to the total score, in points, averaged over the windows (the four segments add up to the score). Young articles have tiny MASE denominators, so a few of them outweigh thousands of established ones. **Compare models column by column**: the gains of the learners over the baseline should show up in the young-article columns.

In [ ]:
seg_rows = []
for (m, c), F in forecasts.items():
    if m in COMPONENTS:
        continue
    seg_rows.append({'model': m, 'cutoff': c, **segment_points(c, postprocess(F, contexts[c], POSTPROCESS))})
seg = pd.DataFrame(seg_rows).groupby('model')[AGE_SEGMENTS].mean()
seg['total'] = seg.sum(axis=1)
counts = pd.DataFrame([pd.Series(article_segments(c)).value_counts() for c in cutoffs]).mean().round(0)
seg.loc['articles per window'] = counts.reindex(AGE_SEGMENTS).to_numpy().tolist() + [counts.reindex(AGE_SEGMENTS).sum()]
display(seg.loc[[*summary.index, 'articles per window']].round(1))

## 15. Overfitting check

Each model's score on the weeks it was trained on vs on unseen weeks. Some gap is normal, because the periods differ in difficulty; the reference models have no training, so **their gap is the period difference alone**. A learner overfits if its gap is clearly bigger: the `excess gap` column.

In [ ]:
ins = results[results.variant == 'in-sample'].groupby('model')['all'].mean()
gap = pd.DataFrame({'trained-on weeks': ins, 'unseen weeks': post.groupby('model')['all'].mean()})
gap['gap'] = gap.iloc[:, 0] - gap.iloc[:, 1]
gap['excess gap vs baseline'] = gap['gap'] - gap.loc[BASELINE, 'gap']
display(gap.loc[summary.index].round(2))

## 16. Which training sources help? (optional, slow)

LightGBM only, same windows, with different `TRAIN_SOURCES`. Set `RUN_SOURCE_ABLATION = True` to rerun it (~45 min). Results from our run, LightGBM score averaged over the 4 windows (post-processed, rounded):

| Training sources | window 2023-01-16 |
|---|---|
| test shop + A, C, D | **−3.35** |
| test shop + A, B, C, D | −3.64 |

Adding Set B made the score slightly worse, as DATASETS.md predicted (it is synthetic and much smoother than the test shop). Only the most recent window was compared, so treat it as a hint, not proof: set `RUN_SOURCE_ABLATION = True` to compare all four windows. Set B stays in the panel, and adding `'B'` to `TRAIN_SOURCES` (§12) switches it on for training.

In [ ]:
RUN_SOURCE_ABLATION = False
SOURCE_SETS = {
    'test shop only': ['test_shop'],
    'test shop + A, C, D': ['test_shop', 'A', 'C', 'D'],
    'test shop + A, B, C, D': ['test_shop', 'A', 'B', 'C', 'D'],
}

if RUN_SOURCE_ABLATION:
    abl = []
    for label, srcs in SOURCE_SETS.items():
        for c in cutoffs:
            ctx = make_context(frame, c, ORIGIN_STRIDE, YOUNG_WEEKS, srcs)
            F = postprocess(fit_lightgbm(ctx, **MODELS['lightgbm'][1])[0], ctx, POSTPROCESS)
            abl.append({'training sources': label, 'cutoff': c.date(), **mase_score(c, F)})
            print(abl[-1])
            del ctx
    abl = pd.DataFrame(abl)
    display(abl.pivot(index='training sources', columns='cutoff', values='all').assign(
        mean=lambda d: d.mean(axis=1)).round(2))

## 17. LightGBM feature importance

Gain importance from the most recent window's model: how much each feature reduced the L1 loss.

In [ ]:
if 'lightgbm' in fitted:
    m = fitted['lightgbm'].model
    imp = pd.Series(m.feature_importance('gain'), index=m.feature_name()).sort_values()
    imp = (imp / imp.sum() * 100).tail(20)
    new_feats = ('launch_', 'oh_', 'stock_', 'sale_share_since_launch', 'n_sale_weeks')
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(imp.index, imp.values, height=0.6, color=[BLUE if i.startswith(new_feats) else GREY for i in imp.index])
    ax.grid(axis='y', visible=False)
    ax.legend(handles=[Patch(color=BLUE, label='new: launch / stock features'), Patch(color=GREY, label='other features')],
              frameon=False, loc='lower right')
    ax.set_xlabel('% of total gain')
    ax.set_title('LightGBM: top 20 features by gain (most recent window)')
    plt.tight_layout(); plt.show()

## 18. Final fit and submission

Refit on **all** history (forecast origin 2023-04-17) and write `article_id, w1 … w13` for 2023-04-24 → 2023-07-17.

`FINAL_MODEL` is the non-reference model with the best mean score over **all** backtest windows. With scores moving 25 points between windows, one window is too noisy to choose on; the price is that the chosen model's test-window score is slightly optimistic. Set it by hand to override.

In [ ]:
candidates = [m for m in summary.index if m not in REFERENCES + COMPONENTS]
FINAL_MODEL = summary.loc[candidates, ('all windows', 'all')].idxmax()
SAVE = True

members = ENSEMBLES[FINAL_MODEL][1] if FINAL_MODEL in ENSEMBLES else [FINAL_MODEL]
print(f'Final model: {FINAL_MODEL} | all windows: {summary.loc[FINAL_MODEL, ("all windows", "all")]} '
      f'(baseline {summary.loc[BASELINE, ("all windows", "all")]})')

ctx_final = make_context(frame, None, ORIGIN_STRIDE, YOUNG_WEEKS, TRAIN_SOURCES)
print(f'Training rows: {len(ctx_final["train"]):,} | forecast origin: {ctx_final["cutoff"].date()}')
final_fc = {}
for m in members:
    t0 = time.perf_counter()
    fit, kw = MODELS[m]
    final_fc[m] = fit(ctx_final, **kw)[0]
    print(f'  fitted {m} ({time.perf_counter() - t0:.0f}s)')

F_final = ENSEMBLES[FINAL_MODEL][0](np.stack([final_fc[m] for m in members]), axis=0) \
    if FINAL_MODEL in ENSEMBLES else final_fc[FINAL_MODEL]
F_final = postprocess(F_final, ctx_final, POSTPROCESS)
sub = make_submission(F_final)

print(f'Articles forecast 0 for all 13 weeks: {(F_final.sum(axis=1) == 0).sum():,} of {len(sub):,}')
print(f'Mean forecast per article-week: {F_final.mean():.2f} (last 13 weeks of history: {Y.iloc[:, -13:].to_numpy().mean():.2f})')
display(sub.head())

if SAVE:
    SUBMISSION_DIR.mkdir(exist_ok=True)
    path = SUBMISSION_DIR / f'submission_full_{re.sub(r"[^A-Za-z0-9]+", "_", FINAL_MODEL).strip("_")}.csv'
    sub.to_csv(path, index=False)
    print('Saved', path)

## 19. All model scores

Mean score over the **test windows**, all 13 weeks, post-processed. Higher is better.

- **Blue**: our learners and ensembles. **Grey**: reference forecasts in the same backtest.
- **Outlined**: the site's published scores, measured on the real hidden weeks (w1–w7). That is a different period from the backtest, so they show the scale, not a like-for-like comparison.

In [ ]:
test_scores = summary[('test windows', 'all')].rename(index=lambda m: f'{m} (final)' if m == FINAL_MODEL else m)
bars = pd.concat([test_scores, pd.Series(SITE_SCORES)]).sort_values()
kind = ['site' if m in SITE_SCORES else 'reference' if m in REFERENCES else 'ours' for m in bars.index]

fig, ax = plt.subplots(figsize=(10, 0.42 * len(bars) + 1.4))
for y, (m, v, k) in enumerate(zip(bars.index, bars.values, kind)):
    style = {'ours': dict(color=BLUE), 'reference': dict(color=GREY),
             'site': dict(facecolor='white', edgecolor=INK, linewidth=1.2)}[k]
    ax.barh(y, v, height=0.62, **style)
    ax.text(v - 0.8, y, f'{v:.1f}', va='center', ha='right', color=INK, fontsize=9)
ax.set_yticks(range(len(bars)), bars.index)
ax.axvline(0, color=INK, linewidth=0.8)
ax.set_xlim(bars.min() * 1.12, max(bars.max(), 0) + 2)
ax.grid(axis='y', visible=False)
ax.legend(handles=[Patch(color=BLUE, label='our models'), Patch(color=GREY, label='reference forecasts (same backtest)'),
                   Patch(facecolor='white', edgecolor=INK, label='site scores (real hidden weeks)')],
          frameon=False, loc='upper left')
ax.set_xlabel('score = (1 − MASE) × 100, higher is better')
ax.set_title('Model scores: mean of the test windows')
plt.tight_layout(); plt.show()